# T11 — The real PR-RT amplicon test

**Question answered.** Random-window fragmentation (used elsewhere) does not simulate what a
real HIV-1 drug-resistance genotype actually is: a *fixed* amplicon, protease codons 1-99
(~297 nt) plus RT codons 1-~240 (~720 nt), always at the same HXB2 coordinates. This notebook
maps every sequence onto HXB2 coordinates, extracts the real clinical regions (PR only, RT
1-240 only, PR+RT combined, and PR+RT with a simulated 200 nt internal dropout for partial
amplification failure), and evaluates every model on these four clinically real inputs.

**One thing this notebook needs that is NOT in your existing Kaggle dataset: the HXB2
reference genome.** It is fetched automatically at runtime from NCBI (accession **K03455.1**)
using Biopython's `Entrez.efetch` -- you do not manually download anything, but you must turn
**Internet: On** in the notebook's settings (Settings -> Internet) for this one cell to work.
If internet is off, this notebook cannot run (there is no dataset workaround for the reference
genome fetch).

**GPU needed.** No for the alignment/extraction step (CPU only). Add GPU only if you also
want to re-run the three CNNs on the four PR/RT inputs below (optional; the Linear SVM and
Random Forest are the main comparators shown here and are what the reviewer specifically
asks for in T11).

**Runtime — the bottleneck is pairwise alignment.** Aligning ~24,072 sequences (median
~3,012 nt) against the ~3,300 nt HXB2 pol reference with Biopython's C-accelerated
`PairwiseAligner`, parallelized across CPU cores, takes roughly **1.5-4 hours** depending on
Kaggle's CPU allocation that session. A `SUBSAMPLE_N` config below lets you first run the
whole notebook on a small subsample (e.g. 500 sequences) in a couple of minutes to check
everything works, then set it to `None` for the full run. If you are tight on the 12 h wall,
run the alignment step in the background with `n_jobs=-1` and check the incremental save
(`t11_hxb2_coords.csv`) partway through -- it resumes-friendly if you save/reload.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and fetch HXB2 (the one external download this test needs)

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, time
import numpy as np, pandas as pd
from multiprocessing import Pool, cpu_count

from Bio import Entrez, SeqIO
from Bio.Align import PairwiseAligner, substitution_matrices
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

# NCBI requires an email for Entrez -- put yours here (any value works, it's just etiquette/rate-limiting)
Entrez.email = "your_email@example.com"

handle = Entrez.efetch(db="nucleotide", id="K03455.1", rettype="gb", retmode="text")
hxb2_record = SeqIO.read(handle, "genbank")
handle.close()
hxb2_seq = str(hxb2_record.seq)
print("HXB2 length:", len(hxb2_seq))

# Standard HXB2 pol coordinates (1-based, GenBank K03455.1 numbering)
PR_START, PR_END = 2253, 2549          # protease, codons 1-99
RT_START, RT_END = 2550, 3269          # RT, codons 1-~240
POL_START, POL_END = 2085, 5096        # full pol (gag-pol reading frame region used for pol gene)

hxb2_pr  = hxb2_seq[PR_START-1:PR_END]
hxb2_rt  = hxb2_seq[RT_START-1:RT_END]
hxb2_pol = hxb2_seq[POL_START-1:POL_END]
print("PR len:", len(hxb2_pr), "| RT len:", len(hxb2_rt), "| pol region len:", len(hxb2_pol))


### 2. Config

In [ ]:

SUBSAMPLE_N = 500   # set to None for the full dataset once you've verified the pipeline works
N_JOBS = max(1, cpu_count() - 1)


### 3. Load data

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
if SUBSAMPLE_N is not None:
    df = df.groupby("subtype", group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(2, SUBSAMPLE_N // df["subtype"].nunique())), random_state=0)
    ).reset_index(drop=True)
    print(f"Subsampled to {len(df)} sequences for a quick end-to-end check.")

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
grp_all = df["cluster"].values
print(f"{len(df)} sequences | {n_cls} classes")


### 4. Align each sequence to HXB2 pol and extract PR / RT by mapped coordinate

In [ ]:

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("NUC.4.4")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

def map_and_extract(seq):
    """Global-align `seq` to the HXB2 pol reference region, then use the alignment's
    coordinate mapping to cut out the PR and RT sub-regions at their HXB2-relative offsets."""
    try:
        aln = aligner.align(hxb2_pol, seq)[0]
        # aligned coordinate pairs: hxb2_pol index -> seq index
        hxb2_aln, seq_aln = aln.aligned
        # build a lookup: hxb2_pol position -> query seq position (nearest, via aligned blocks)
        pos_map = {}
        for (h0, h1), (s0, s1) in zip(hxb2_aln, seq_aln):
            for offset in range(h1 - h0):
                pos_map[h0 + offset] = s0 + offset

        pr_rel_start, pr_rel_end = PR_START - POL_START, PR_END - POL_START
        rt_rel_start, rt_rel_end = RT_START - POL_START, RT_END - POL_START

        def extract(rel_start, rel_end):
            idxs = [pos_map[i] for i in range(rel_start, rel_end) if i in pos_map]
            if len(idxs) < (rel_end - rel_start) * 0.5:
                return None  # too much missing -> unreliable extraction
            return seq[min(idxs):max(idxs)+1]

        pr = extract(pr_rel_start, pr_rel_end)
        rt = extract(rt_rel_start, rt_rel_end)
        return pr, rt
    except Exception:
        return None, None

def worker(seq):
    pr, rt = map_and_extract(seq)
    return pr, rt

print(f"Aligning {len(X_all)} sequences to HXB2 with {N_JOBS} workers (this is the slow step)...")
t0 = time.time()
with Pool(N_JOBS) as pool:
    results = pool.map(worker, list(X_all), chunksize=20)
print(f"done in {time.time()-t0:.0f}s")

pr_seqs = [r[0] for r in results]
rt_seqs = [r[1] for r in results]
n_ok = sum(1 for p, r in zip(pr_seqs, rt_seqs) if p and r)
print(f"Successfully extracted both PR and RT for {n_ok}/{len(X_all)} sequences "
      f"({100*n_ok/len(X_all):.1f}%). Sequences that fail extraction are dropped below.")

coords_df = pd.DataFrame({"idx": range(len(X_all)), "subtype": df["subtype"].values,
                           "cluster": grp_all, "pr": pr_seqs, "rt": rt_seqs})
coords_df.to_csv(f"{OUT}/t11_hxb2_coords.csv", index=False)


### 5. Build the four clinically real inputs

In [ ]:

import random as _random

def dropout_200nt(seq, rng):
    """Simulate a partial amplification failure: delete a random 200 nt internal window."""
    if len(seq) <= 220:
        return seq
    start = rng.integers(50, len(seq) - 200 - 50)
    return seq[:start] + seq[start+200:]

good = coords_df.dropna(subset=["pr", "rt"]).reset_index(drop=True)
rng = np.random.default_rng(0)

good["pr_only"]        = good["pr"]
good["rt_only"]        = good["rt"]
good["pr_rt_combined"] = good["pr"] + good["rt"]
good["pr_rt_dropout"]  = good["pr_rt_combined"].apply(lambda s: dropout_200nt(s, rng))

print(good[["pr_only","rt_only","pr_rt_combined","pr_rt_dropout"]].applymap(len).describe())


### 6. Evaluate Linear SVM (and Random Forest) on each of the four real inputs, cluster-disjoint split

In [ ]:

y_good = LabelEncoder().fit_transform(good["subtype"].values)
grp_good = good["cluster"].values

s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
trv, te = next(s1.split(good.index.values, y_good, grp_good))

INPUTS = ["pr_only", "rt_only", "pr_rt_combined", "pr_rt_dropout"]
rows = []
for col in INPUTS:
    seq_tr = good.iloc[trv][col].values
    seq_te = good.iloc[te][col].values
    ytr, yte = y_good[trv], y_good[te]

    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xte = vec.transform(seq_te)

    for model_name, clf in [
        ("linear_svm", LinearSVC(class_weight="balanced", random_state=0, max_iter=20000)),
        ("random_forest", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                                   random_state=0, n_jobs=-1)),
    ]:
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        rows.append(dict(input_region=col, model=model_name,
                          f1=f1_score(yte, pred, average="macro"),
                          bal_acc=balanced_accuracy_score(yte, pred),
                          n_train=len(trv), n_test=len(te)))
        print(rows[-1])

t11_table = pd.DataFrame(rows)
t11_table.to_csv(f"{OUT}/t11_real_amplicon_results.csv", index=False)
t11_table


### 7. Interpretation cell

In [ ]:

print("""
If the Linear SVM holds macro-F1 >= 0.95 on 'pr_rt_combined' (the standard clinical
drug-resistance genotype), that is a deployable-tool claim for the Discussion (Section 4.5/4.8).
Compare pr_rt_dropout to pr_rt_combined to see how much a realistic partial-amplification
failure costs -- report both numbers, not just the clean case.
""")
print(t11_table.pivot(index="model", columns="input_region", values="f1").round(4))


### 8. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r t11_real_amplicon_outputs.zip t11_hxb2_coords.csv t11_real_amplicon_results.csv